# 09 · Prescriptif — plan de rétention optimal sous budget
Pour chaque client et action : gain = P(attrition) × uplift × CLV − coût.
Optimisation exacte (MILP) comparée à une heuristique gloutonne.

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
from pyspark.sql import functions as F
from churn.dbx import get_config
from churn import prescriptive as P

cfg = get_config(dbutils)
dbutils.widgets.text("budget_eur", str(cfg.campaign_budget_eur))       # 2 000 € par défaut
budget = float(dbutils.widgets.get("budget_eur"))

preds = spark.table(cfg.t("attrition_predictions"))
snapshot = preds.agg(F.max("snapshot_date")).first()[0]
scored = preds.where(F.col("snapshot_date") == snapshot).toPandas()
scored["clv"] = P.customer_value(scored)
print(f"{len(scored)} clients scorés au {snapshot} | budget {budget:,.0f} €")

In [0]:
plan = P.optimize_campaign(scored, budget, method="milp")
greedy = P.optimize_campaign(scored, budget, method="greedy")
print(f"MILP    : {len(plan)} clients, coût {plan.cost.sum():,.0f} €, gain net espéré {plan.net_gain.sum():,.0f} €")
print(f"Glouton : {len(greedy)} clients, coût {greedy.cost.sum():,.0f} €, gain net espéré {greedy.net_gain.sum():,.0f} €")

plan = plan.merge(scored[["client_id", "segment", "segment_comportemental", "risk_band"]], on="client_id")
plan["snapshot_date"] = snapshot
out = spark.createDataFrame(plan).withColumn("created_at", F.current_timestamp())
table = cfg.t("retention_campaign")
if not spark.catalog.tableExists(table):
    out.write.saveAsTable(table)
else:
    (out.write.mode("overwrite").option("replaceWhere", f"snapshot_date = '{snapshot}'")
     .option("mergeSchema", "true").saveAsTable(table))

display(plan.groupby(["segment_comportemental", "action"])
        .agg(clients=("client_id", "count"), cout=("cost", "sum"), gain=("net_gain", "sum")).round(0))